# Load UniProt official registry

In [1]:
from collections import defaultdict
from pathlib import Path

import bioregistry as br
import requests

In [2]:
params = {"format": "json", "query": "*", "size": 500}
response = requests.get("https://rest.uniprot.org/database/search", params=params)
response.raise_for_status()
registry_data = response.json()
uniprot_dblist = registry_data["results"]
uniprot_dblist

[{'name': 'ABCD curated depository of sequenced antibodies',
  'id': 'DB-0236',
  'abbrev': 'ABCD',
  'linkType': 'Explicit',
  'servers': ['https://web.expasy.org/abcd'],
  'dbUrl': 'https://web.expasy.org/cgi-bin/abcd/search_abcd.pl?input=%u',
  'category': 'Protocols and materials databases',
  'statistics': {'reviewedProteinCount': 3196, 'unreviewedProteinCount': 619}},
 {'name': 'The Alliance of Genome Resources',
  'id': 'DB-0266',
  'abbrev': 'AGR',
  'pubMedId': '31552413',
  'doiId': '10.1093/nar/gkz813',
  'linkType': 'Explicit',
  'servers': ['https://www.alliancegenome.org/'],
  'dbUrl': 'https://www.alliancegenome.org/gene/%s',
  'category': 'Organism-specific databases',
  'statistics': {'reviewedProteinCount': 68617,
   'unreviewedProteinCount': 246243}},
 {'name': 'Agora',
  'id': 'DB-0283',
  'abbrev': 'Agora',
  'linkType': 'Explicit',
  'servers': ['https://agora.adknowledgeportal.org'],
  'dbUrl': 'https://agora.adknowledgeportal.org/genes/%s/',
  'category': 'Misce

In [3]:
uniprot_dblist_db_names = {
    entry["name"].strip().lower() for entry in registry_data["results"] if isinstance(entry, dict) and entry.get("name")
}

print("Official UniProt name count:", len(uniprot_dblist_db_names))

uniprot_dblist_prefixes = {
    entry["abbrev"].strip().lower()
    for entry in registry_data["results"]
    if isinstance(entry, dict) and entry.get("abbrev")
}

print("Official UniProt abbrev count:", len(uniprot_dblist_prefixes))
uniprot_dblist_prefixes

Official UniProt name count: 185
Official UniProt abbrev count: 185


{'abcd',
 'agora',
 'agr',
 'allergome',
 'alphafolddb',
 'antibodypedia',
 'antifam',
 'arachnoserver',
 'araport',
 'bgee',
 'bindingdb',
 'biocyc',
 'biogrid',
 'biogrid-orcs',
 'biomuta',
 'bmrb',
 'brenda',
 'carbonyldb',
 'card',
 'cazy',
 'ccds',
 'cd-code',
 'cdd',
 'cgd',
 'chembl',
 'chitars',
 'civic',
 'clingen',
 'clinpgx',
 'collectf',
 'complexportal',
 'conoserver',
 'corum',
 'cptac',
 'cptc',
 'ctd',
 'dbsnp',
 'ddbj',
 'depod',
 'dictybase',
 'dip',
 'disgenet',
 'disprot',
 'dmdm',
 'dnasu',
 'drugbank',
 'drugcentral',
 'echobase',
 'eggnog',
 'elm',
 'embl',
 'emdb',
 'ensembl',
 'ensemblbacteria',
 'ensemblfungi',
 'ensemblmetazoa',
 'ensemblplants',
 'ensemblprotists',
 'enzyme',
 'esther',
 'euhcvdb',
 'evolutionarytrace',
 'expressionatlas',
 'flybase',
 'funcoup',
 'funfam',
 'genatlas',
 'genbank',
 'gencc',
 'gene3d',
 'genecards',
 'geneid',
 'genereviews',
 'genetree',
 'genewiki',
 'genomernai',
 'glyconnect',
 'glycosmos',
 'glygen',
 'go',
 'gpcrdb',
 

## Load ID mapping prefixes.txt

List of all unique prefixes that appear in the UniProt ID mapping data file: 
[idmapping.dat.gz](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/idmapping/idmapping.dat.gz)

In [4]:
ID_MAPPING_PREFIXES = Path("data/prefixes.txt")
idmapping_prefixes = {line.strip().lower() for line in ID_MAPPING_PREFIXES.read_text().split("\n") if line.strip()}
print("UniProt ID mapping prefixes:", len(idmapping_prefixes))
idmapping_prefixes

UniProt ID mapping prefixes: 103


{'allergome',
 'arachnoserver',
 'araport',
 'biocyc',
 'biogrid',
 'biomuta',
 'ccds',
 'cgd',
 'chembl',
 'chitars',
 'collectf',
 'complexportal',
 'conoserver',
 'cptac',
 'crc64',
 'dictybase',
 'dip',
 'disprot',
 'dmdm',
 'dnasu',
 'drugbank',
 'echobase',
 'eggnog',
 'embl',
 'embl-cds',
 'emdb',
 'ensembl',
 'ensembl_pro',
 'ensembl_trs',
 'ensemblgenome',
 'ensemblgenome_pro',
 'ensemblgenome_trs',
 'esther',
 'euhcvdb',
 'flybase',
 'gene_name',
 'gene_orderedlocusname',
 'gene_orfname',
 'gene_synonym',
 'genecards',
 'geneid',
 'genereviews',
 'genetree',
 'genewiki',
 'genomernai',
 'gi',
 'glyconnect',
 'guidetopharmacology',
 'hgnc',
 'hogenom',
 'ideal',
 'japonicusdb',
 'kegg',
 'legiolist',
 'leproma',
 'maizegdb',
 'merops',
 'mgi',
 'mim',
 'mint',
 'ncbi_taxid',
 'nextprot',
 'oma',
 'opentargets',
 'orphanet',
 'orthodb',
 'patric',
 'pdb',
 'peroxibase',
 'pharmgkb',
 'phi-base',
 'plantreactome',
 'pombase',
 'proteomicsdb',
 'pseudocap',
 'reactome',
 'rebase'

In [5]:
## Prefixes in the ID mapping file but not in UniProt official list
idmapping_not_in_dblist = idmapping_prefixes - uniprot_dblist_prefixes
print(len(idmapping_not_in_dblist))
print(sorted(idmapping_not_in_dblist))

25
['crc64', 'embl-cds', 'ensembl_pro', 'ensembl_trs', 'ensemblgenome', 'ensemblgenome_pro', 'ensemblgenome_trs', 'gene_name', 'gene_orderedlocusname', 'gene_orfname', 'gene_synonym', 'gi', 'ncbi_taxid', 'nextprot', 'pharmgkb', 'refseq_nt', 'treefam', 'uniparc', 'uniprotkb-id', 'uniref100', 'uniref50', 'uniref90', 'wbparasite_trs_pro', 'wormbase_pro', 'wormbase_trs']


### Classification of ID Mapping Prefixes Not Present in UniProt Official Cross-Reference Registry

The following prefixes appear in the ID mapping-derived set but are not listed in the UniProt official cross-reference registry.

They fall into several categories:

1.	Internal UniProt metadata
2.	Subtype mappings 
3.	External biological databases
4.	Deprecated or taxonomy identifiers
5.	UniProt-derived resources

In [6]:
## create a dictionary with empty lists
classification = defaultdict(list)

SUBTYPE_MAPPING = {
    "embl-cds",  # EMBL CDS subtype
    "refseq_nt",  # RefSeq nucleotide subtype
}

for p in sorted(idmapping_not_in_dblist):
    # internal annotation fields
    if p.startswith("gene_") or p in {"crc64", "uniprotkb-id"}:
        classification["internal_metadata"].append(p)

    # UniProt derived databases
    elif p.startswith("uniref") or p == "uniparc":
        classification["uniprot_derived_db"].append(p)

    # subtype-specific identifiers
    elif p in SUBTYPE_MAPPING or any(token in p for token in ["_pro", "_trs"]):
        classification["subtype_mapping"].append(p)

    # deprecated identifiers
    elif p == "gi":
        classification["deprecated_identifier"].append(p)

    # taxonomy identifiers
    elif p == "ncbi_taxid":
        classification["taxonomy_identifier"].append(p)

    # external database candidate
    else:
        classification["external_database_candidate"].append(p)

for k, v in classification.items():
    print(f"\n{k} ({len(v)}):")
    print(sorted(v))


internal_metadata (6):
['crc64', 'gene_name', 'gene_orderedlocusname', 'gene_orfname', 'gene_synonym', 'uniprotkb-id']

subtype_mapping (9):
['embl-cds', 'ensembl_pro', 'ensembl_trs', 'ensemblgenome_pro', 'ensemblgenome_trs', 'refseq_nt', 'wbparasite_trs_pro', 'wormbase_pro', 'wormbase_trs']

external_database_candidate (4):
['ensemblgenome', 'nextprot', 'pharmgkb', 'treefam']

deprecated_identifier (1):
['gi']

taxonomy_identifier (1):
['ncbi_taxid']

uniprot_derived_db (4):
['uniparc', 'uniref100', 'uniref50', 'uniref90']


## Load parquet prefixes

The following code must be run on BERDL.

In [7]:
from berdl_notebook_utils.setup_spark_session import get_spark_session
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower

In [8]:
spark = get_spark_session("PrefixExploration")

In [12]:
# Load the identifier table from the uniprot raw dataset on BERDL
# this dataset has already been partitioned by `db` column to make it quicker to access
df = spark.read.parquet(
    "s3a://cdm-lake/tenant-general-warehouse/kbase/datasets/uniprot/uniprot_kb/identifier_partitioned"
)
df.printSchema()
# get the distinct prefixes from the identifier table
identifier_table_dbs = {d.asDict().get("db") for d in df.select("db").distinct().collect()}
print("Databases in the `identifiers` table:")
identifier_table_dbs

root
 |-- entity_id: string (nullable = true)
 |-- db: string (nullable = true)
 |-- xref: string (nullable = true)
 |-- description: string (nullable = true)
 |-- _dlt_load_id: string (nullable = true)
 |-- _dlt_id: string (nullable = true)
 |-- relationship: string (nullable = true)

Databases in the `identifiers` table:


{'ABCD',
 'AGR',
 'Agora',
 'Allergome',
 'AlphaFoldDB',
 'AntiFam',
 'Antibodypedia',
 'ArachnoServer',
 'Araport',
 'BMRB',
 'BRENDA',
 'Bgee',
 'BindingDB',
 'BioCyc',
 'BioGRID',
 'BioGRID-ORCS',
 'BioMuta',
 'CARD',
 'CAZy',
 'CCDS',
 'CD-CODE',
 'CDD',
 'CGD',
 'CIViC',
 'CORUM',
 'CPTAC',
 'CPTC',
 'CTD',
 'CarbonylDB',
 'ChEMBL',
 'ChiTaRS',
 'ClinPGx',
 'CollecTF',
 'ComplexPortal',
 'ConoServer',
 'DEPOD',
 'DIP',
 'DMDM',
 'DNASU',
 'DisGeNET',
 'DisProt',
 'DrugBank',
 'DrugCentral',
 'EC',
 'ELM',
 'EMDB',
 'ESTHER',
 'EchoBASE',
 'EnsemblBacteria',
 'EnsemblFungi',
 'EnsemblMetazoa',
 'EnsemblPlants',
 'EnsemblProtists',
 'EvolutionaryTrace',
 'ExpressionAtlas',
 'FlyBase',
 'FunCoup',
 'FunFam',
 'GO',
 'Gene3D',
 'GeneCards',
 'GeneID',
 'GeneReviews',
 'GeneTree',
 'GeneWiki',
 'GenomeRNAi',
 'GlyConnect',
 'GlyCosmos',
 'GlyGen',
 'Gramene',
 'GuidetoPHARMACOLOGY',
 'HAMAP',
 'HGNC',
 'HOGENOM',
 'HPA',
 'IDEAL',
 'IMGT_GENE-DB',
 'InParanoid',
 'IntAct',
 'InterPro',

In [15]:
# save the identifier table db prefixes to file for easier querying without needing BERDL access
outfile = Path("data") / "identifier_table_prefixes.txt"
outfile.write_text("\n".join(sorted(identifier_table_dbs)))

1421

In [23]:
parquet_set = {db.lower() for db in identifier_table_dbs}
print("Parquet prefixes:", len(parquet_set))
print(parquet_set)

Parquet prefixes: 171
{'ensemblprotists', 'peroxibase', 'strenda-db', 'civic', 'swisslipids', 'funfam', 'mgi', 'pro', 'corum', 'sfld', 'hogenom', 'biocyc', 'peptideatlas', 'echobase', 'drugbank', 'tuberculist', 'allergome', 'carbonyldb', 'prints', 'antibodypedia', 'smr', 'supfam', 'phosphositeplus', 'jpost', 'chitars', 'reactome', 'tair', 'expressionatlas', 'ncbitaxon', 'genewiki', 'esther', 'genbank', 'pumba', 'genereviews', 'panther', 'rgd', 'proteomes', 'malacards', 'moondb', 'flybase', 'araport', 'hpa', 'agora', 'ensemblbacteria', 'reproduction-2dpage', 'cgd', 'funcoup', 'disgenet', 'sgd', 'inparanoid', 'ncbifam', 'glyconnect', 'ycharos', 'moonprot', 'phylomedb', 'pombase', 'dmdm', 'prosite', 'hamap', 'pir', 'orphanet', 'swisspalm', 'dictybase', 'ensemblfungi', 'glycosmos', 'japonicusdb', 'clinpgx', 'bindingdb', 'glygen', 'interpro', 'vgnc', 'arachnoserver', 'ensemblmetazoa', 'sabio-rk', 'string', 'gene3d', 'sasbdb', 'ccds', 'drugcentral', 'abcd', 'legiolist', 'opentargets', 'metos

In [26]:
## Prefixes in parquet but not in UniProt official list
parquet_not_in_uniprot = parquet_set - uniprot_dblist_prefixes
print(
    f"Prefixes in the Identifier table that don't appear in the UniProt dblist prefixes ({len(parquet_not_in_uniprot)}):"
)
print(sorted(parquet_not_in_uniprot))

Prefixes in the Identifier table that don't appear in the UniProt dblist prefixes (3):
['ec', 'ncbitaxon', 'uniprot']


### Interpretation

These are not true registry gaps:

- **ec** – Represents EC numbers. 
- **ncbitaxon** – A naming variation of NCBI Taxonomy.
- **uniprot** – UniProt itself is not listed as an external cross-reference database.

Conclusion: No external databases detected


In [27]:
external_candidates = classification["external_database_candidate"]

In [28]:
for p in external_candidates:
    in_name = p in uniprot_dblist_db_names
    in_abbrev = p in uniprot_dblist_prefixes
    in_parquet = p in parquet_set
    print(f"{p:20} | name={in_name} | abbrev={in_abbrev}| parquet={in_parquet}")

ensemblgenome        | name=False | abbrev=False| parquet=False
nextprot             | name=False | abbrev=False| parquet=False
pharmgkb             | name=False | abbrev=False| parquet=False
treefam              | name=False | abbrev=False| parquet=False


Some prefixes classified as external database candidates (e.g., nextprot, pharmgkb, treefam) do not currently appear in the BERDL parquet dataset.

This indicates that while these namespaces correspond to real biological databases, they are not used in the current dataset snapshot. They remain classified as external databases based on their semantic meaning rather than dataset usage.

# Classification of Differences

### A. UniProt annotation metadata 
- crc64
- gene_name
- gene_orderedlocusname
- gene_orfname
- gene_synonym
- uniprotkb-id

These fields represent internal UniProt annotations rather than cross-references to external databases. Examples include gene name annotations and sequence checksums maintained directly within UniProt records.


### B. UniProt derived databases 
- uniparc 
- uniref100
- uniref50
- uniref90

These are UniProt internal resources, no need remapping. 


### C. Internal NCBI identifiers
- gi
- ncbi_taxid
  
ncbi_taxid is taxonomy identifier,
gi used by NCBI but has been officially deprecated.



### D. Database subtype mappings 
- embl-cds
- refseq_nt
- ensembl_pro
- ensembl_trs
- ensemblgenome_pro
- ensemblgenome_trs
- wormbase_pro
- wormbase_trs
- wbparasite_trs_pro

#### patterns:
- `_pro` → protein identifiers
- `_trs` → transcript identifiers
- `_cds` → coding sequence identifiers
- `_nt` → nucleotide accessions

These indicate the identifier type within a parent database. Need normalize to parent database prefix.


### E. External database 

- ensemblgenome
- nextprot
- pharmgkb
- treefam

#### examples

- `EnsemblGenome` – genome annotation database
- `NextProt` – human protein knowledgebase
- `PharmGKB` – pharmacogenomics database
- `TreeFam` – phylogenetic gene family database

In [29]:
prefixes = sorted(idmapping_not_in_dblist)
results = []

for p in prefixes:
    resource = br.get_resource(p)
    normalized = br.normalize_prefix(p)

    results.append({"prefix": p, "bioregistry_found": resource is not None, "normalized": normalized})

for r in results:
    print(r)

{'prefix': 'crc64', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'embl-cds', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'ensembl_pro', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'ensembl_trs', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'ensemblgenome', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'ensemblgenome_pro', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'ensemblgenome_trs', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'gene_name', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'gene_orderedlocusname', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'gene_orfname', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'gene_synonym', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'gi', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'ncbi_taxid', 'bioregistry_found': True, 'normalized': 'ncbitaxon'}
{'prefix': 'nextprot', 'bio

## conclusion 

The Bioregistry package partially support prefix remapping, but it is not sufficient as a solution for the UniProt / BERDL prefix governance workflow.

### Bioregistry package effective for: 

- Canonical prefix normalization
- Synonym resolution (e.g., ncbi_taxid → ncbitaxon)
- Validation of recognized external biological databases

### Bioregistry does not cover: 

- Subtype-specific identifiers (e.g., ensembl_pro, refseq_nt)
- UniProt internal metadata fields (e.g., gene_name, crc64)
- UniProt-derived internal resources (e.g., uniref100)
- Deprecated identifiers, requires manual handling or exclusion (e.g., gi)

In [30]:
INTERNAL_PREFIXES = set(classification.get("internal_metadata", []))

## subtype has feature, xxx_pro/xxx_trs/xxx_nt/xxx_cds
SUBTYPE_TOKENS = {"pro", "trs", "nt", "cds"}


def infer_parent_prefix(prefix: str) -> str:
    """Deduce the parent prefix from the suffix.

    :param prefix: prefix string
    :type prefix: str
    :return: parent prefix
    :rtype: str
    """
    tokens = prefix.replace("-", "_").split("_")
    tokens = [t for t in tokens if t not in SUBTYPE_TOKENS]
    return "_".join(tokens)


SUBTYPE_RULES = {p: infer_parent_prefix(p) for p in classification.get("subtype_mapping", [])}

SUBTYPE_RULES

{'embl-cds': 'embl',
 'ensembl_pro': 'ensembl',
 'ensembl_trs': 'ensembl',
 'ensemblgenome_pro': 'ensemblgenome',
 'ensemblgenome_trs': 'ensemblgenome',
 'refseq_nt': 'refseq',
 'wbparasite_trs_pro': 'wbparasite',
 'wormbase_pro': 'wormbase',
 'wormbase_trs': 'wormbase'}

In [36]:
## INTERNAL_PREFIXES:
## if prefix.startswith("gene_")
## if prefix in {"crc64"}
## if prefix.endswith("-id")

INTERNAL_KEYWORDS = {
    "crc64",  ## only need UniProt checksum, not namespace
}


def is_internal_prefix(prefix: str) -> bool:
    prefix = prefix.lower()

    if prefix.startswith("gene_"):
        return True

    if prefix in INTERNAL_KEYWORDS:
        return True

    return bool(prefix.endswith("-id"))

In [32]:
INTERNAL_PREFIXES = {p for p in idmapping_prefixes if is_internal_prefix(p)}
INTERNAL_PREFIXES

{'crc64',
 'gene_name',
 'gene_orderedlocusname',
 'gene_orfname',
 'gene_synonym',
 'uniprotkb-id'}

In [33]:
DEPRECATED_PREFIXES = set(classification.get("deprecated_identifier", []))

In [34]:
def remap_prefix(prefix: str) -> dict:
    prefix = prefix.lower()

    if is_internal_prefix(prefix):
        return {"original": prefix, "canonical": None, "source": "internal"}

    if prefix in DEPRECATED_PREFIXES:
        return {"original": prefix, "canonical": None, "source": "deprecated"}

    if prefix in SUBTYPE_RULES:
        return {"original": prefix, "canonical": SUBTYPE_RULES[prefix], "source": "subtype"}

    normalized = br.normalize_prefix(prefix)
    if normalized:
        return {"original": prefix, "canonical": normalized, "source": "bioregistry"}

    return {"original": prefix, "canonical": None, "source": "unresolved"}

In [77]:
import pandas as pd

results = [remap_prefix(p) for p in sorted(idmapping_prefixes)]
df = pd.DataFrame(results)
df.head(25)

,original,canonical,source
0,allergome,allergome,bioregistry
1,arachnoserver,arachnoserver,bioregistry
2,araport,araport,bioregistry
3,biocyc,biocyc,bioregistry
4,biogrid,biogrid,bioregistry
5,biomuta,NaN,unresolved
6,ccds,ccds,bioregistry
7,cgd,cgd,bioregistry
8,chembl,chembl,bioregistry
9,chitars,NaN,unresolved


In [78]:
print(df["source"].value_counts())

source
bioregistry    56
unresolved     31
subtype         9
internal        6
deprecated      1
Name: count, dtype: int64


In [82]:
unresolved_df = df[df["source"] == "unresolved"].sort_values("original").drop(columns=["canonical"])
unresolved_df

,original,source
5,biomuta,unresolved
9,chitars,unresolved
10,collectf,unresolved
13,cptac,unresolved
18,dmdm,unresolved
19,dnasu,unresolved
23,embl,unresolved
29,ensemblgenome,unresolved
32,esther,unresolved
33,euhcvdb,unresolved


#### These are UniProt cross-reference databases. 

Uniprot cluster resources not in bioregistry. 

In [ ]:
## The mappings that already made when prefixes are existing in Bioregistry

print(br.normalize_prefix("tair"))
print(br.normalize_prefix("patric"))
print(br.normalize_prefix("oma"))
print(br.normalize_prefix("merops"))

None
None
None
None


These prefixes are for external biological databases not covered by the Bioregistry.

### Final Evaluation

Bioregistry was evaluated for prefix remapping.
- It directly recognizes 56 out of 103 observed prefixes;
- It correctly normalizes prefix variants; 
- 31 prefixes are not covered by Bioregistry; these correspond mainly to UniProt-specific resources.

After incorporating subtype rules, internal metadata handling and deprecated identifiers, ~70% of prefixes can be governed.
